In [1]:
"""
===============================================================================
RELATÓRIO TÉCNICO DE DECISIONING SD-WAN ZERO-TRUST
===============================================================================
Arquivo: desafio_ac1_master_sdwan.py
Topologia: 12 Roteadores (Origem: Nó 0, Destino: Nó 11)
Semente Estocástica: np.random.seed(2026)

1. Análise de Segurança dos Nós (Zero-Trust):
   - Nós Penalizados (Reputação < 50):
     * Nó 1 (Reputação: 36/100) -> Inseguro (Risco de Interceptação/Comprometimento)
     * Nó 9 (Reputação: 32/100) -> Inseguro (Risco de Interceptação/Comprometimento)
   - Nós Selecionados no Caminho Seguro:
     * Nó 0 (Origem, Reputação: 100/100)
     * Nó 7 (Roteador Intermediário, Reputação: 99/100)
     * Nó 11 (Destino, Reputação: 100/100)

2. Rota Selecionada pelo Algoritmo Evolutivo:
   - Rota Ótima Encontrada: [0 -> 7 -> 11]
   - Métricas do Enlace:
     * Latência Total: 82.00 ms
     * Perda de Pacotes Total: 1.43 %
     * Penalidade de Segurança (P_Seguranca): 0.0
     * Fitness Ponderado Final: 96.25

3. Justificativa de Desvio e Seleção de Rota:
   O motor de decisão Zero-Trust aplica uma penalidade estática pesada (P_Seguranca = 5000)
   a qualquer rota que incorpore os nós 1 ou 9. O algoritmo evolutivo desviou com sucesso
   desses nós não confiáveis, selecionando o Nó 7 como o salto intermediário ideal por apresentar
   excelente reputação (99/100) e a menor combinação ponderada de latência e perda.
===============================================================================
"""

import numpy as np

np.random.seed(2026)

NUM_NOS = 12
ORIGEM = 0
DESTINO = 11

W1 = 1.0
W2 = 10.0

TAM_POP = 80
GERACOES = 150
TAXA_MUTACAO = 0.2
USAR_ELITISMO = True

reputacao_nos = np.random.randint(30, 100, size=NUM_NOS)
reputacao_nos[ORIGEM] = 100
reputacao_nos[DESTINO] = 100

matriz_latencia = np.random.uniform(10, 80, (NUM_NOS, NUM_NOS))
matriz_perda = np.random.uniform(0.1, 5.0, (NUM_NOS, NUM_NOS))

np.fill_diagonal(matriz_latencia, 0)
np.fill_diagonal(matriz_perda, 0)
matriz_latencia[ORIGEM, DESTINO] = np.inf
matriz_perda[ORIGEM, DESTINO] = np.inf

def calcular_fitness(rota, matriz_lat, matriz_perda, reputacao, w1=W1, w2=W2):
    if len(rota) < 2 or rota[0] != ORIGEM or rota[-1] != DESTINO:
        return 1e9, 1e9, 1e9, 5000.0

    latencia_total = 0.0
    perda_total = 0.0
    penalidade_seguranca = 0.0

    for no in rota:
        if reputacao[no] < 50:
            penalidade_seguranca = 5000.0
            break

    for i in range(len(rota) - 1):
        l = matriz_lat[rota[i], rota[i+1]]
        p = matriz_perda[rota[i], rota[i+1]]
        if np.isinf(l) or np.isinf(p):
            return 1e9, 1e9, 1e9, 5000.0
        latencia_total += l
        perda_total += p

    fitness = w1 * latencia_total + w2 * perda_total + penalidade_seguranca
    return fitness, latencia_total, perda_total, penalidade_seguranca

nos_intermediarios = [n for n in range(1, NUM_NOS - 1)]
populacao = []

for _ in range(TAM_POP):
    tamanho_rota = np.random.randint(1, 4)
    meio = list(np.random.choice(nos_intermediarios, tamanho_rota, replace=False))
    populacao.append([ORIGEM] + meio + [DESTINO])

for g in range(GERACOES):
    avaliacoes = [calcular_fitness(r, matriz_latencia, matriz_perda, reputacao_nos) for r in populacao]
    custos = [a[0] for a in avaliacoes]
    melhor_idx = np.argmin(custos)

    novos = []
    if USAR_ELITISMO:
        novos.append(populacao[melhor_idx].copy())

    while len(novos) < TAM_POP:
        i1, i2 = np.random.choice(TAM_POP, 2, replace=False)
        pai = populacao[i1] if custos[i1] < custos[i2] else populacao[i2]

        filho = pai.copy()
        if np.random.rand() < TAXA_MUTACAO:
            sub = filho[1:-1]
            if len(sub) > 0:
                idx = np.random.randint(0, len(sub))
                sub[idx] = np.random.choice(nos_intermediarios)
                filho = [ORIGEM] + list(dict.fromkeys(sub)) + [DESTINO]

        novos.append(filho)

    populacao = novos

avaliacoes_finais = [calcular_fitness(r, matriz_latencia, matriz_perda, reputacao_nos) for r in populacao]
custos_finais = [a[0] for a in avaliacoes_finais]
melhor_idx = np.argmin(custos_finais)

melhor_rota = populacao[melhor_idx]
fit, lat, perda, pen = avaliacoes_finais[melhor_idx]

print(f"[SD-WAN Decisioning] Rota Selecionada: {melhor_rota}")
print(f"Fitness: {fit:.2f} | Latência Total: {lat:.2f} ms | Perda Pacotes: {perda:.2f}% | Penalidade Seg.: {pen:.1f}")

[SD-WAN Decisioning] Rota Selecionada: [0, np.int64(7), 11]
Fitness: 96.25 | Latência Total: 82.00 ms | Perda Pacotes: 1.43% | Penalidade Seg.: 0.0
